In [ ]:
Celda 1: Cargar los datos guardados

In [1]:
from pyspark.sql import SparkSession

# 1. Iniciamos la sesión de Spark
spark = SparkSession.builder.appName("Semana12_Regresion_Final").getOrCreate()

# 2. Leemos el archivo Parquet que dejó Lizette en la Semana 10
ruta_datos = "/home/jovyan/work/semanas/Semana 10/modelos/datos_etiquetados_kmeans"
df_clusters = spark.read.parquet(ruta_datos)

# 3. Mostramos las primeras 5 filas para verificar que todo exista
df_clusters.select("marca", "precio_kg", "rating", "opiniones").show(5)

+-----+-----------------+-----------------+---------+
|marca|        precio_kg|           rating|opiniones|
+-----+-----------------+-----------------+---------+
|    0|9.020000457763672|4.800000190734863|      414|
|    1|6.860000133514404|4.699999809265137|      100|
|    3|5.150000095367432|4.699999809265137|      241|
|    1|5.349999904632568|4.699999809265137|      155|
|    0|7.639999866485596|4.800000190734863|      341|
+-----+-----------------+-----------------+---------+
only showing top 5 rows



In [ ]:
Celda 2: Crear los vectores y escalar (Sección 11.4.1)

In [2]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# 1. Empaquetamos 'rating' y 'opiniones' en un solo vector
assembler_regresion = VectorAssembler(
    inputCols=["rating", "opiniones"],
    outputCol="features_regresion"
)
df_vector_reg = assembler_regresion.transform(df_clusters)

# 2. Escalamos los datos para homogeneizar las magnitudes numéricas
scaler_reg = StandardScaler(inputCol="features_regresion", outputCol="scaledFeatures_regresion")
scaler_model_reg = scaler_reg.fit(df_vector_reg)
df_para_regresion = scaler_model_reg.transform(df_vector_reg)

# 3. Definimos nuestra variable a predecir renombrando 'precio_kg' a 'label_precio'
df_para_regresion = df_para_regresion.withColumnRenamed("precio_kg", "label_precio")
df_para_regresion = df_para_regresion.drop("prediction")

# 4. Dividimos en 70% entrenamiento y 30% prueba
train_reg, test_reg = df_para_regresion.randomSplit([0.7, 0.3], seed=42)

print(f"Registros para entrenar: {train_reg.count()}")
print(f"Registros para evaluar: {test_reg.count()}")

Registros para entrenar: 679
Registros para evaluar: 236


In [ ]:
Celda 3: Entrenar el algoritmo de Regresión Lineal (Sección 11.4.2)

In [3]:
from pyspark.ml.regression import LinearRegression

# 1. Configuramos el modelo de regresión lineal de Spark
lr_regresion = LinearRegression(
    featuresCol="scaledFeatures_regresion",
    labelCol="label_precio",
    maxIter=10
)

# 2. Ajustamos (entrenamos) el modelo con el set de entrenamiento
lr_reg_model = lr_regresion.fit(train_reg)

# 3. Predecimos los precios en el set de prueba
predictions_regresion = lr_reg_model.transform(test_reg)

# 4. Desplegamos la tabla comparativa de resultados
print("=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===")
predictions_regresion.select("marca", "label_precio", "prediction").show(10)

=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===
+-----+------------------+-----------------+
|marca|      label_precio|       prediction|
+-----+------------------+-----------------+
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|7.317322158572727|
|    0|12.569999694824219|7.317322158572727|
|    0|12.569999694824219|7.317322158572727|
|    0|12.569999694824219|7.317322158572727|
+-----+------------------+-----------------+
only showing top 10 rows



In [ ]:
Celda 4: Calcular métricas de error y ver la ecuación matemática (Sección 11.4.3)

In [4]:
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Definimos los evaluadores de calidad (R2 y RMSE)
evaluator_r2 = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="r2")
evaluator_rmse = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="rmse")

r2 = evaluator_r2.evaluate(predictions_regresion)
rmse = evaluator_rmse.evaluate(predictions_regresion)

print("==================================================")
print("   EVALUACIÓN MATEMÁTICA DE LA REGRESIÓN")
print("==================================================")
print(f"R² (Coeficiente de Determinación): {r2 * 100:.2f}%")
print(f"RMSE (Desviación promedio en dinero): {rmse:.4f}")
print("==================================================")

# 2. Coeficientes de la ecuación final
print(f"Intersección (Precio base fijo): {lr_reg_model.intercept:.4f}")
print(f"Pendiente de 'rating': {lr_reg_model.coefficients[0]:.4f}")
print(f"Pendiente de 'opiniones': {lr_reg_model.coefficients[1]:.4f}")

   EVALUACIÓN MATEMÁTICA DE LA REGRESIÓN
R² (Coeficiente de Determinación): 5.68%
RMSE (Desviación promedio en dinero): 2.8363
Intersección (Precio base fijo): -14.6993
Pendiente de 'rating': 0.6410
Pendiente de 'opiniones': 0.0421


In [ ]:
c

In [5]:
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Definimos los evaluadores de calidad (R2 y RMSE)
evaluator_r2 = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="r2")
evaluator_rmse = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="rmse")

r2 = evaluator_r2.evaluate(predictions_regresion)
rmse = evaluator_rmse.evaluate(predictions_regresion)

print("==================================================")
print("   EVALUACIÓN MATEMÁTICA DE LA REGRESIÓN")
print("==================================================")
print(f"R² (Coeficiente de Determinación): {r2 * 100:.2f}%")
print(f"RMSE (Desviación promedio en dinero): {rmse:.4f}")
print("==================================================")

# 2. Coeficientes de la ecuación final
print(f"Intersección (Precio base fijo): {lr_reg_model.intercept:.4f}")
print(f"Pendiente de 'rating': {lr_reg_model.coefficients[0]:.4f}")
print(f"Pendiente de 'opiniones': {lr_reg_model.coefficients[1]:.4f}")

   EVALUACIÓN MATEMÁTICA DE LA REGRESIÓN
R² (Coeficiente de Determinación): 5.68%
RMSE (Desviación promedio en dinero): 2.8363
Intersección (Precio base fijo): -14.6993
Pendiente de 'rating': 0.6410
Pendiente de 'opiniones': 0.0421
